Here we shall see some examples of how to use the `homology_representation` function, including a comparison of runtimes against the code of BRR, and how to use it to compute details of theta characteristics. 

First we just need to load in the file. If you have changed the relative location of `home_reps.py`, you will need to make the relevant adjustment in the line below. 

In [1]:
load("hom_reps.py")

First let's test that the code gets the correct answer for a hyperelliptic curve

In [2]:
# Initialise the group and its generating vector 
G = CyclicPermutationGroup(2)
iota = G.gen()
sigma = 2
gv = 2*(sigma+1)*[iota]
# Compute the homology representation for iota
hr = homology_representation(G, gv, BR=ZZ, elms=[iota])
# Confirm that the homology representation is -identity. 
hr[0] == -matrix.identity(2*sigma)

True

Now let's see how we can compare runtime with the previous method of BRR. 

In [4]:
# Load the code of BRR
load("Companion_Files/polyB.sage")

# Create a function to package in the method of BRR 
def homology_representation_BRR(G, gv, BR=ZZ):
    P = Poly(G, gv)
    return [P._representation(g).change_ring(BR) for g in P.generators]

# Load the function to time the computation. 
from time import time

# Initialise values to be used throughout the loop
all_t_pairs_F2 = []
BR = FiniteField(2)

# Loop over all the generating vectors for each genus
for g in range(2, 13):
    gdata = load("Companion_Files/invariant_generation_data/invariant_generation_data_g={}.sobj".format(g))
    for di in gdata:
        # Extract the gv 
        G = PermutationGroup(gap_group = gap.Image(gap.IsomorphismPermGroup(gap.SmallGroup(*di[1]))))
        gv = di[3]
        t = []
        # For each method time the computation. 
        for hr in [homology_representation, homology_representation_BRR]:
            try:
                ct = time()
                P = hr(G, gv, BR=BR)
                t.append(time() - ct)
            except (ValueError, AttributeError) as e:
                print("Unexplained error with data {}".format(di[:-2]))
                continue
        # Provided no errors occured, add both times to the data
        if len(t) == 2:
            all_t_pairs_F2.append(t)
            
# Find the ratio of the runtimes and take their geometric mean. 
proportional_runtime = 10^(sum([log(ts[0]/ts[1], 10) for ts in all_t_pairs_F2]).n()/len(all_t_pairs_F2))
proportional_runtime

ver 2020.07.24


0.799046663255901

Finally we verify that the number of invariant characteristics of the modular curve $X(p)$ is 1 for $p=7, 11, 13, 17$ and find that their parity is always even. 

In [5]:
# We load the method to compute affine subspaces as we shall intersect the invariant subspaces for 
# each generator
from sage.geometry.hyperplane_arrangement.affine_subspace import AffineSubspace

# parity computes the parity of a binary vector, which shall be required later. 
def parity(v):
    return sum([v[i]*v[i+g] for i in range(g)])

# Initialise values to be used throughout the loop
F2 = FiniteField(2)
BR = F2

# Loop over the relevant range of primes. 
for p in primes(7, 18):
    g = ZZ((p+2)*(p-3)*(p-5)/24)
    print("p:", p)
    print("genus:", g)

    # Initialise the group
    G = PSL(2, p)
    GO = G.order()
    c = [p, 3, 2]
    I2g = matrix.identity(2*g, F2)
    
    # Use the method of BRR to find a generating vector
    gv = find_generating_vector(G, c)
    try:
        P = homology_representation(G, gv, BR=BR, intersection_matrix=True, elms=gv[:-1])
    except (ValueError, AttributeError) as e:
        print("Unexplained error with data {}".format((p, g, ngen)))
        continue

    # Use the method of Disney-Hogg, following Kallel and Sjerve, to compute the number
    # of invariant characteristics and their parity. 
    IM = P[-1]
    F, C = IM.symplectic_form()
    Mlist = [(C*L*C.inverse()).dense_matrix() for L in P[0]]
    Vbars = [vector([sum([A[j,i]*A[j+g,i] for j in range(g)]) 
                     for i in range(2*g)], F2) for A in Mlist]
    MIs = [M-I2g for M in Mlist]
    X0s = [MI.solve_left(-V) for MI, V in zip(MIs, Vbars)]
    AS = [AffineSubspace(X0, MI.kernel()) for X0, MI in zip(X0s, MIs)] 
    Sols = AS.pop()
    while Sols and AS:
        AI = AS.pop()
        Sols = Sols.intersection(AI)
    nIC = 0
    if Sols:
        nIC = 2^Sols.dimension()

    print("# invariant characteristics: {}".format(nIC))
    print("parity:", parity(Sols.point()))

p: 7
genus: 3
# invariant characteristics: 1
parity: 0
